# Ablation Study -- WELFake Fake News Detection

This notebook quantifies the contribution of each branch
in the hybrid model by systematically removing one branch
at a time and measuring the resulting performance drop.

The ablation study directly answers the question:
"Does each branch actually contribute, or could a simpler
2-branch model achieve the same performance?"

Three variants are trained and evaluated:
Variant A (No Linguistic): TF-IDF + GloVe/BiLSTM only
Variant B (No LSTM)      : TF-IDF + Linguistic only
Variant C (No TF-IDF)    : GloVe/BiLSTM + Linguistic only

All variants use the same optimal hyperparameters found
in notebook 07 to ensure fair comparison. The only
variable is which branches are present.

Full hybrid baseline: F1 Macro = 0.9843 (notebook 07)

Pipeline:
1. Load all feature tracks (same as notebook 07)
2. Define 3 ablation model architectures
3. Train each variant with notebook 07 optimal params
4. Evaluate all 3 variants on test set
5. Generate 5 comparison plots
6. Save results and artifacts

## Section 1 -- Setup and Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import gc
import os
os.environ['OMP_NUM_THREADS'] = '1'
import re
import sys
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    f1_score as sk_f1,
    roc_curve,
    roc_auc_score,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score
)
import joblib

BASE          = '/content/drive/MyDrive/MSU Semester 4/Applied ML/Project'
PROCESSED_DIR = BASE + '/processed/'
SRC_DIR       = BASE + '/src/'
MODELS_DIR    = BASE + '/models/'
FIGURES_DIR   = BASE + '/figures/ablation/'
RESULTS_DIR   = BASE + '/results/'

os.makedirs(MODELS_DIR,  exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

sys.path.append(SRC_DIR)

from models   import train_model, save_pytorch_model, get_device
from evaluate import compute_metrics
from utils    import set_seed, print_section

set_seed(42)
device = get_device()
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

# Optimal hyperparameters from notebook 07
# These are fixed for all ablation variants to ensure
# the only variable is which branches are present
OPTIMAL_LR      = 0.001
OPTIMAL_LSTM    = 256
OPTIMAL_DROPOUT = 0.5
OPTIMAL_FREEZE  = False
BATCH_SIZE      = 32
MAX_LEN         = 300
EMBEDDING_DIM   = 100
FULL_HYBRID_F1  = 0.9843

print(f"Device         : {device}")
print(f"PyTorch version: {torch.__version__}")
gpu_name = os.popen(
    'nvidia-smi --query-gpu=name --format=csv,noheader'
).read().strip()
print(f"GPU            : {gpu_name}")
print()
print("Optimal hyperparameters from notebook 07:")
print(f"  lr={OPTIMAL_LR}, lstm={OPTIMAL_LSTM}, "
      f"dropout={OPTIMAL_DROPOUT}, freeze={OPTIMAL_FREEZE}")

## Section 2 -- Load All Feature Tracks

All three feature tracks are loaded using the same
pipeline as notebook 07. TF-IDF is kept sparse to
avoid the ~10 GB RAM spike from dense conversion.
Linguistic features are loaded from pre-saved arrays.

In [ ]:
print_section("Loading All Feature Tracks")

train_df = pd.read_csv(PROCESSED_DIR + 'train_clean.csv')
val_df   = pd.read_csv(PROCESSED_DIR + 'val_clean.csv')
test_df  = pd.read_csv(PROCESSED_DIR + 'test_clean.csv')

X_train_text = train_df['content'].fillna('').tolist()
X_val_text   = val_df['content'].fillna('').tolist()
X_test_text  = test_df['content'].fillna('').tolist()

y_train = train_df['label'].values
y_val   = val_df['label'].values
y_test  = test_df['label'].values

# --- Track A: TF-IDF (sparse) ---
# Refit with optimal params from notebook 03 and 07
# Kept sparse to avoid ~10 GB RAM spike
tfidf_vec = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.90,
    sublinear_tf=True,
    strip_accents='unicode',
    token_pattern=r'\w{2,}'
)
print("Fitting TF-IDF...")
X_train_tfidf = tfidf_vec.fit_transform(X_train_text)
X_val_tfidf   = tfidf_vec.transform(X_val_text)
X_test_tfidf  = tfidf_vec.transform(X_test_text)
TFIDF_DIM     = X_train_tfidf.shape[1]
print(f"TF-IDF shape : {X_train_tfidf.shape}")

gc.collect()
torch.cuda.empty_cache()

# --- Track B: Sequences ---
word2idx   = joblib.load(MODELS_DIR + 'tokenizer.joblib')
VOCAB_SIZE = len(word2idx)

def tokenize(text):
    """Tokenize matching notebook 04 and 07 exactly."""
    return re.sub(r"[^\w\s']", '', text.lower()).split()

def encode_and_pad(texts, word2idx, max_len):
    """
    Convert texts to padded integer sequences.
    Must match tokenization in notebooks 04 and 07
    for consistent vocabulary index mapping.
    """
    sequences = []
    for text in texts:
        tokens = tokenize(text)
        ids    = [word2idx.get(t, 1) for t in tokens]
        ids    = ids[:max_len]
        ids    = ids + [0] * (max_len - len(ids))
        sequences.append(ids)
    return np.array(sequences, dtype=np.int64)

print("Encoding sequences...")
X_train_seq = encode_and_pad(X_train_text, word2idx, MAX_LEN)
X_val_seq   = encode_and_pad(X_val_text,   word2idx, MAX_LEN)
X_test_seq  = encode_and_pad(X_test_text,  word2idx, MAX_LEN)
print(f"Sequences shape : {X_train_seq.shape}")

# Load GloVe embedding matrix from notebook 05
glove_matrix = np.load(MODELS_DIR + 'glove_embedding_matrix.npy')
print(f"GloVe matrix    : {glove_matrix.shape}")

# --- Track C: Linguistic features ---
X_train_ling = np.load(PROCESSED_DIR + 'X_train_linguistic.npy')
X_val_ling   = np.load(PROCESSED_DIR + 'X_val_linguistic.npy')
X_test_ling  = np.load(PROCESSED_DIR + 'X_test_linguistic.npy')
LINGUISTIC_DIM = X_train_ling.shape[1]
print(f"Linguistic shape: {X_train_ling.shape}")

## Section 3 -- Ablation Model Architectures

Three 2-branch model classes are defined, each removing
one branch from the full 3-branch hybrid. The fusion
dimension changes accordingly:

Full hybrid    : 128 + 128 + 32 = 288 dims
No linguistic  : 128 + 128      = 256 dims
No LSTM        : 128       + 32 = 160 dims
No TF-IDF      :       128 + 32 = 160 dims

All other layer sizes remain identical to ensure
that performance differences are due only to the
missing branch, not to architectural differences.

In [ ]:
class NoLinguisticHybrid(nn.Module):
    """
    2-branch hybrid: TF-IDF branch + GloVe/BiLSTM branch.
    Linguistic features branch removed to test its contribution.
    Fusion dimension: 128 + 128 = 256.

    Parameters
    ----------
    tfidf_dim : int
        Dimension of TF-IDF input features.
    vocab_size : int
        Vocabulary size for embedding layer.
    embedding_dim : int
        Word embedding dimension.
    lstm_units : int
        BiLSTM hidden units per direction.
    dropout : float
        Dropout rate in fusion layer.
    """
    def __init__(self, tfidf_dim, vocab_size,
                 embedding_dim, lstm_units, dropout):
        super().__init__()

        # TF-IDF branch: same as full hybrid branch 1
        self.branch1 = nn.Sequential(
            nn.Linear(tfidf_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU()
        )

        # GloVe + BiLSTM branch: same as full hybrid branch 2
        self.embedding = nn.Embedding(
            vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=lstm_units,
            bidirectional=True,
            batch_first=True
        )
        self.linear_lstm = nn.Linear(lstm_units * 2, 128)

        # Fusion: 128 + 128 = 256 (no linguistic 32)
        self.fusion = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )

    def forward(self, x_tfidf, x_seq):
        """Forward pass with TF-IDF and sequence inputs only."""
        out1 = self.branch1(x_tfidf)

        embedded  = self.embedding(x_seq)
        lstm_out, _ = self.lstm(embedded)
        # Max pooling over sequence dimension captures
        # the most salient feature across all positions
        out2 = torch.relu(self.linear_lstm(
            torch.max(lstm_out, dim=1).values))

        # Concatenate 2 branches (no linguistic branch)
        combined = torch.cat([out1, out2], dim=1)
        return torch.sigmoid(self.fusion(combined))


class NoLSTMHybrid(nn.Module):
    """
    2-branch hybrid: TF-IDF branch + Linguistic branch.
    GloVe/BiLSTM branch removed to test its contribution.
    Fusion dimension: 128 + 32 = 160.

    Parameters
    ----------
    tfidf_dim : int
    linguistic_dim : int
    dropout : float
    """
    def __init__(self, tfidf_dim, linguistic_dim, dropout):
        super().__init__()

        # TF-IDF branch: identical to full hybrid
        self.branch1 = nn.Sequential(
            nn.Linear(tfidf_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU()
        )

        # Linguistic branch: identical to full hybrid
        self.branch3 = nn.Sequential(
            nn.Linear(linguistic_dim, 32),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # Fusion: 128 + 32 = 160 (no LSTM 128)
        self.fusion = nn.Sequential(
            nn.Linear(160, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )

    def forward(self, x_tfidf, x_ling):
        """Forward pass with TF-IDF and linguistic inputs only."""
        out1 = self.branch1(x_tfidf)
        out3 = self.branch3(x_ling)

        # Concatenate 2 branches (no LSTM branch)
        combined = torch.cat([out1, out3], dim=1)
        return torch.sigmoid(self.fusion(combined))


class NoTFIDFHybrid(nn.Module):
    """
    2-branch hybrid: GloVe/BiLSTM branch + Linguistic branch.
    TF-IDF branch removed to test its contribution.
    Fusion dimension: 128 + 32 = 160.

    Parameters
    ----------
    vocab_size : int
    embedding_dim : int
    lstm_units : int
    linguistic_dim : int
    dropout : float
    """
    def __init__(self, vocab_size, embedding_dim,
                 lstm_units, linguistic_dim, dropout):
        super().__init__()

        # GloVe + BiLSTM branch: identical to full hybrid
        self.embedding = nn.Embedding(
            vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=lstm_units,
            bidirectional=True,
            batch_first=True
        )
        self.linear_lstm = nn.Linear(lstm_units * 2, 128)

        # Linguistic branch: identical to full hybrid
        self.branch3 = nn.Sequential(
            nn.Linear(linguistic_dim, 32),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # Fusion: 128 + 32 = 160 (no TF-IDF 128)
        self.fusion = nn.Sequential(
            nn.Linear(160, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )

    def forward(self, x_seq, x_ling):
        """Forward pass with sequence and linguistic inputs only."""
        embedded    = self.embedding(x_seq)
        lstm_out, _ = self.lstm(embedded)
        out2 = torch.relu(self.linear_lstm(
            torch.max(lstm_out, dim=1).values))
        out3 = self.branch3(x_ling)

        # Concatenate 2 branches (no TF-IDF branch)
        combined = torch.cat([out2, out3], dim=1)
        return torch.sigmoid(self.fusion(combined))


print("All 3 ablation model classes defined.")

## Section 4 -- Dataset Classes for Each Variant

Each ablation variant needs its own Dataset class
because the number and type of inputs differs.
TF-IDF is kept sparse and densified lazily in
__getitem__ to prevent RAM overflow, matching
the approach used in notebook 07.

In [ ]:
class NoLingDataset(Dataset):
    """Dataset for NoLinguisticHybrid: TF-IDF + sequences."""
    def __init__(self, tfidf_sparse, sequences, labels):
        self.tfidf_sparse = tfidf_sparse
        self.sequences    = torch.tensor(
            sequences, dtype=torch.long)
        self.labels       = torch.tensor(
            labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # Densify one TF-IDF row at a time to save RAM
        tfidf_row = torch.tensor(
            self.tfidf_sparse[idx].toarray().squeeze(),
            dtype=torch.float32)
        return self.sequences[idx], tfidf_row, self.labels[idx]


class NoLSTMDataset(Dataset):
    """Dataset for NoLSTMHybrid: TF-IDF + linguistic."""
    def __init__(self, tfidf_sparse, linguistic, labels):
        self.tfidf_sparse = tfidf_sparse
        self.linguistic   = torch.tensor(
            linguistic, dtype=torch.float32)
        self.labels       = torch.tensor(
            labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        tfidf_row = torch.tensor(
            self.tfidf_sparse[idx].toarray().squeeze(),
            dtype=torch.float32)
        return tfidf_row, self.linguistic[idx], self.labels[idx]


class NoTFIDFDataset(Dataset):
    """Dataset for NoTFIDFHybrid: sequences + linguistic."""
    def __init__(self, sequences, linguistic, labels):
        self.sequences  = torch.tensor(
            sequences, dtype=torch.long)
        self.linguistic = torch.tensor(
            linguistic, dtype=torch.float32)
        self.labels     = torch.tensor(
            labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (self.sequences[idx],
                self.linguistic[idx],
                self.labels[idx])


print("All 3 dataset classes defined.")

## Section 5 -- Training Helper

A reusable training function handles all three variants.
Early stopping with patience 3 prevents overfitting,
matching the final training setup in notebook 07.

In [ ]:
def train_ablation_variant(model, train_loader,
                            val_loader, variant_name):
    """
    Train an ablation model variant with early stopping.

    Uses the same optimal hyperparameters found in
    notebook 07 to ensure fair comparison across variants.
    The only variable is which branches are present.

    Parameters
    ----------
    model : nn.Module
        Ablation model instance.
    train_loader : DataLoader
        Training data loader for this variant.
    val_loader : DataLoader
        Validation data loader for this variant.
    variant_name : str
        Name for logging output.

    Returns
    -------
    tuple
        (trained_model, history_dict)
    """
    model     = model.to(device)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=OPTIMAL_LR)
    criterion = nn.BCELoss()

    best_val_loss  = float('inf')
    best_weights   = None
    no_improve     = 0
    history        = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [],  'val_acc': []
    }

    print(f"\nTraining {variant_name}...")

    for epoch in range(10):
        # Training pass
        model.train()
        train_losses, train_correct = [], 0

        for batch in train_loader:
            # Unpack all inputs except last (label)
            *inputs, labels = batch
            inputs = [x.to(device) for x in inputs]
            labels = labels.to(device)

            optimizer.zero_grad()
            preds = model(*inputs).squeeze()
            loss  = criterion(preds, labels)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())
            train_correct += (
                (preds > 0.5).float() == labels
            ).sum().item()

        # Validation pass
        model.eval()
        val_losses, val_correct = [], 0

        with torch.no_grad():
            for batch in val_loader:
                *inputs, labels = batch
                inputs = [x.to(device) for x in inputs]
                labels = labels.to(device)
                preds  = model(*inputs).squeeze()
                loss   = criterion(preds, labels)
                val_losses.append(loss.item())
                val_correct += (
                    (preds > 0.5).float() == labels
                ).sum().item()

        train_loss = np.mean(train_losses)
        val_loss   = np.mean(val_losses)
        train_acc  = train_correct / len(train_loader.dataset)
        val_acc    = val_correct   / len(val_loader.dataset)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        print(f"  Epoch {epoch+1:2d}/10  "
              f"train_loss={train_loss:.4f}  "
              f"val_loss={val_loss:.4f}  "
              f"val_acc={val_acc:.4f}")

        # Early stopping on validation loss
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights  = {
                k: v.clone()
                for k, v in model.state_dict().items()
            }
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= 3:
                print(f"  Early stopping at epoch {epoch+1}.")
                break

    # Restore best weights
    model.load_state_dict(best_weights)
    model.eval()
    return model, history


def evaluate_variant(model, test_loader, variant_name):
    """
    Evaluate an ablation variant on the test set.

    Returns predictions, scores, and all metrics.
    Test set is never used during training to ensure
    unbiased performance estimates.

    Parameters
    ----------
    model : nn.Module
        Trained ablation model.
    test_loader : DataLoader
        Test data loader for this variant.
    variant_name : str
        Name for labeling results.

    Returns
    -------
    tuple
        (all_preds, all_scores, all_labels, metrics_dict)
    """
    model.eval()
    all_preds, all_scores, all_labels = [], [], []

    with torch.no_grad():
        for batch in test_loader:
            *inputs, labels = batch
            inputs = [x.to(device) for x in inputs]
            scores = model(*inputs).squeeze().cpu().numpy()
            preds  = (scores > 0.5).astype(int)
            all_scores.extend(scores.tolist())
            all_preds.extend(preds.tolist())
            all_labels.extend(
                labels.numpy().astype(int).tolist())

    metrics = compute_metrics(
        all_labels, all_preds, all_scores, variant_name)
    print(f"\n{variant_name} Test Results:")
    print(f"  F1 Macro : {metrics['F1_Macro']:.4f}")
    print(f"  Accuracy : {metrics['Accuracy']:.4f}")
    print(f"  ROC-AUC  : {metrics['ROC_AUC']:.4f}")
    return all_preds, all_scores, all_labels, metrics


print("Training and evaluation helpers defined.")

## Section 6 -- Load GloVe Weights Helper

In [ ]:
def load_glove_weights(model, glove_matrix):
    """
    Load pretrained GloVe weights into model embedding layer.
    Weights are set to fine-tunable (requires_grad=True)
    matching the optimal setting found in notebook 07.

    Parameters
    ----------
    model : nn.Module
        Model instance with self.embedding layer.
    glove_matrix : np.ndarray
        Pretrained GloVe matrix shape (vocab_size, 100).

    Returns
    -------
    nn.Module
        Model with GloVe weights loaded.
    """
    glove_tensor = torch.tensor(
        glove_matrix, dtype=torch.float32)
    with torch.no_grad():
        model.embedding.weight.copy_(glove_tensor)
    # Fine-tune embeddings (OPTIMAL_FREEZE=False from nb07)
    model.embedding.weight.requires_grad = True
    return model


print("GloVe loader defined.")

## Section 7 -- Variant A: No Linguistic Branch

Removes the linguistic features branch (10d -> 32).
Tests whether writing style features contribute
beyond TF-IDF vocabulary and GloVe semantic features.

Expected: small F1 drop since linguistic features
showed weak discriminative power in notebook 06
(except Flesch readability and word count).

In [ ]:
print_section("Variant A: No Linguistic Branch")

# Build model and load GloVe weights
model_a = NoLinguisticHybrid(
    tfidf_dim=TFIDF_DIM,
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    lstm_units=OPTIMAL_LSTM,
    dropout=OPTIMAL_DROPOUT
)
model_a = load_glove_weights(model_a, glove_matrix)

# DataLoaders for this variant
train_loader_a = DataLoader(
    NoLingDataset(X_train_tfidf, X_train_seq, y_train),
    batch_size=BATCH_SIZE, shuffle=True)
val_loader_a   = DataLoader(
    NoLingDataset(X_val_tfidf, X_val_seq, y_val),
    batch_size=BATCH_SIZE, shuffle=False)
test_loader_a  = DataLoader(
    NoLingDataset(X_test_tfidf, X_test_seq, y_test),
    batch_size=BATCH_SIZE, shuffle=False)

model_a, history_a = train_ablation_variant(
    model_a, train_loader_a, val_loader_a,
    'No Linguistic')

preds_a, scores_a, labels_a, metrics_a = evaluate_variant(
    model_a, test_loader_a, 'No Linguistic')

save_pytorch_model(
    model_a, MODELS_DIR + 'ablation_no_linguistic.pt')

# Free memory before next variant
del model_a
gc.collect()
torch.cuda.empty_cache()

## Section 8 -- Variant B: No LSTM Branch

Removes the GloVe + BiLSTM branch entirely.
Tests whether sequential semantic context contributes
beyond TF-IDF and linguistic writing style features.

Expected: larger F1 drop since BiLSTM captures
patterns TF-IDF bag-of-words cannot (word order,
negation context, semantic similarity).

In [ ]:
print_section("Variant B: No LSTM Branch")

model_b = NoLSTMHybrid(
    tfidf_dim=TFIDF_DIM,
    linguistic_dim=LINGUISTIC_DIM,
    dropout=OPTIMAL_DROPOUT
)

train_loader_b = DataLoader(
    NoLSTMDataset(X_train_tfidf, X_train_ling, y_train),
    batch_size=BATCH_SIZE, shuffle=True)
val_loader_b   = DataLoader(
    NoLSTMDataset(X_val_tfidf, X_val_ling, y_val),
    batch_size=BATCH_SIZE, shuffle=False)
test_loader_b  = DataLoader(
    NoLSTMDataset(X_test_tfidf, X_test_ling, y_test),
    batch_size=BATCH_SIZE, shuffle=False)

model_b, history_b = train_ablation_variant(
    model_b, train_loader_b, val_loader_b,
    'No LSTM')

preds_b, scores_b, labels_b, metrics_b = evaluate_variant(
    model_b, test_loader_b, 'No LSTM')

save_pytorch_model(
    model_b, MODELS_DIR + 'ablation_no_lstm.pt')

del model_b
gc.collect()
torch.cuda.empty_cache()

## Section 9 -- Variant C: No TF-IDF Branch

Removes the TF-IDF sparse feature branch entirely.
Tests whether vocabulary frequency patterns contribute
beyond GloVe semantic embeddings and linguistic features.

Expected: moderate to large F1 drop since TF-IDF
captures domain-specific term frequencies that GloVe
general embeddings may miss.

In [ ]:
print_section("Variant C: No TF-IDF Branch")

model_c = NoTFIDFHybrid(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    lstm_units=OPTIMAL_LSTM,
    linguistic_dim=LINGUISTIC_DIM,
    dropout=OPTIMAL_DROPOUT
)
model_c = load_glove_weights(model_c, glove_matrix)

train_loader_c = DataLoader(
    NoTFIDFDataset(X_train_seq, X_train_ling, y_train),
    batch_size=BATCH_SIZE, shuffle=True)
val_loader_c   = DataLoader(
    NoTFIDFDataset(X_val_seq, X_val_ling, y_val),
    batch_size=BATCH_SIZE, shuffle=False)
test_loader_c  = DataLoader(
    NoTFIDFDataset(X_test_seq, X_test_ling, y_test),
    batch_size=BATCH_SIZE, shuffle=False)

model_c, history_c = train_ablation_variant(
    model_c, train_loader_c, val_loader_c,
    'No TF-IDF')

preds_c, scores_c, labels_c, metrics_c = evaluate_variant(
    model_c, test_loader_c, 'No TF-IDF')

save_pytorch_model(
    model_c, MODELS_DIR + 'ablation_no_tfidf.pt')

del model_c
gc.collect()
torch.cuda.empty_cache()

## Section 10 -- Ablation Results Summary

In [ ]:
print_section("Ablation Results Summary")

# Build ablation results table
ablation_results = [
    {
        'Variant'      : 'No Linguistic (TF-IDF + LSTM)',
        'F1_Macro'     : metrics_a['F1_Macro'],
        'Accuracy'     : metrics_a['Accuracy'],
        'ROC_AUC'      : metrics_a['ROC_AUC'],
        'Delta_vs_Full': round(
            metrics_a['F1_Macro'] - FULL_HYBRID_F1, 4)
    },
    {
        'Variant'      : 'No LSTM (TF-IDF + Linguistic)',
        'F1_Macro'     : metrics_b['F1_Macro'],
        'Accuracy'     : metrics_b['Accuracy'],
        'ROC_AUC'      : metrics_b['ROC_AUC'],
        'Delta_vs_Full': round(
            metrics_b['F1_Macro'] - FULL_HYBRID_F1, 4)
    },
    {
        'Variant'      : 'No TF-IDF (LSTM + Linguistic)',
        'F1_Macro'     : metrics_c['F1_Macro'],
        'Accuracy'     : metrics_c['Accuracy'],
        'ROC_AUC'      : metrics_c['ROC_AUC'],
        'Delta_vs_Full': round(
            metrics_c['F1_Macro'] - FULL_HYBRID_F1, 4)
    },
    {
        'Variant'      : 'Full Hybrid (all 3 branches)',
        'F1_Macro'     : FULL_HYBRID_F1,
        'Accuracy'     : 0.9845,
        'ROC_AUC'      : 0.9991,
        'Delta_vs_Full': 0.0
    }
]

ablation_df = pd.DataFrame(ablation_results)
ablation_df.to_csv(
    RESULTS_DIR + 'ablation_results.csv', index=False)

print(f"{'Variant':<40} {'F1':>7} {'Delta':>8}")
print("-" * 58)
for _, row in ablation_df.iterrows():
    delta_str = f"{row['Delta_vs_Full']:+.4f}"
    print(f"{row['Variant']:<40} "
          f"{row['F1_Macro']:>7.4f} "
          f"{delta_str:>8}")

print()
# Identify most and least important branch
ablation_only = ablation_df[
    ablation_df['Variant'] != 'Full Hybrid (all 3 branches)']
most_important  = ablation_only.loc[
    ablation_only['F1_Macro'].idxmin(), 'Variant']
least_important = ablation_only.loc[
    ablation_only['F1_Macro'].idxmax(), 'Variant']
print(f"Most important branch  : {most_important}")
print(f"Least important branch : {least_important}")

## Section 11 -- Visualizations

Five plots are generated for the report and poster.
All plots follow the same style as notebook 07.

### Plot 24 -- Ablation Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

variant_labels = [
    'No Linguistic\n(TF-IDF+LSTM)',
    'No LSTM\n(TF-IDF+Linguistic)',
    'No TF-IDF\n(LSTM+Linguistic)',
    'Full Hybrid\n(3 branches)'
]
f1_values = [
    metrics_a['F1_Macro'],
    metrics_b['F1_Macro'],
    metrics_c['F1_Macro'],
    FULL_HYBRID_F1
]
# Ablation variants use amber, full hybrid uses coral
# matching the color convention from notebook 07
bar_colors = ['#E9C46A', '#E9C46A', '#E9C46A', '#E76F51']

bars = ax.bar(variant_labels, f1_values,
              color=bar_colors, alpha=0.85,
              edgecolor='white', linewidth=1.2)

# Value labels on top of each bar
for bar, score in zip(bars, f1_values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            score + 0.001,
            f'{score:.4f}',
            ha='center', va='bottom', fontsize=11)

# Delta annotations below each ablation bar
for i, (bar, score) in enumerate(
        zip(bars[:3], f1_values[:3])):
    delta = score - FULL_HYBRID_F1
    ax.text(bar.get_x() + bar.get_width() / 2,
            min(f1_values) - 0.008,
            f'{delta:+.4f}',
            ha='center', va='top', fontsize=9,
            color='#E76F51' if delta < 0 else '#2A9D8F')

# Dashed reference line at full hybrid F1
ax.axhline(y=FULL_HYBRID_F1,
           color='#E76F51', linestyle='--',
           linewidth=1.5, alpha=0.7,
           label=f'Full hybrid ({FULL_HYBRID_F1:.4f})')

ax.set_ylim(max(0.85, min(f1_values) - 0.02), 1.005)
ax.set_xlabel('Model Variant', fontsize=12)
ax.set_ylabel('F1 Macro (Test Set)', fontsize=12)
ax.set_title('Ablation Study -- Branch Contribution',
             fontsize=14)

legend_handles = [
    mpatches.Patch(color='#E9C46A',
                   label='Ablation variants (2 branches)'),
    mpatches.Patch(color='#E76F51',
                   label='Full hybrid (3 branches)'),
]
ax.legend(handles=legend_handles, fontsize=10)

fig.suptitle('Ablation Study -- WELFake Hybrid Model',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR + '24_ablation_barplot.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/ablation/24_ablation_barplot.png")

### Plot 25 -- Ablation vs All Baselines

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))

# All 9 model variants in progression order
all_model_names = [
    'Random\nForest', 'XGBoost',
    'Logistic\nReg.', 'LinearSVC',
    'BiLSTM',
    'No\nLinguistic', 'No\nLSTM', 'No\nTF-IDF',
    'Full\nHybrid'
]
all_f1_scores = [
    0.9429, 0.9627, 0.9703, 0.9720,
    0.9783,
    metrics_a['F1_Macro'],
    metrics_b['F1_Macro'],
    metrics_c['F1_Macro'],
    FULL_HYBRID_F1
]
# Color encodes model family
all_colors = [
    '#E9C46A', '#E9C46A', '#E9C46A', '#E9C46A',
    '#2A9D8F',
    '#E9C46A', '#E9C46A', '#E9C46A',
    '#E76F51'
]

bars = ax.bar(all_model_names, all_f1_scores,
              color=all_colors, alpha=0.85,
              edgecolor='white', linewidth=1.2)

for bar, score in zip(bars, all_f1_scores):
    ax.text(bar.get_x() + bar.get_width() / 2,
            score + 0.001,
            f'{score:.4f}',
            ha='center', va='bottom', fontsize=9)

ax.set_ylim(0.90, 1.012)
ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('F1 Macro (Test Set)', fontsize=12)
ax.set_title('All Models -- Classical, Neural, Hybrid, Ablation',
             fontsize=14)

legend_handles = [
    mpatches.Patch(color='#E9C46A',
                   label='Classical + ablation variants'),
    mpatches.Patch(color='#2A9D8F',
                   label='BiLSTM baseline'),
    mpatches.Patch(color='#E76F51',
                   label='Full hybrid (best)'),
]
ax.legend(handles=legend_handles, fontsize=10)

fig.suptitle('Complete Model Comparison -- WELFake',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR + '25_ablation_vs_baselines.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/ablation/25_ablation_vs_baselines.png")

### Plot 26 -- Confusion Matrices for Ablation Variants

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
class_labels = ['Fake (0)', 'Real (1)']

variant_data = [
    ('No Linguistic', preds_a, labels_a),
    ('No LSTM',       preds_b, labels_b),
    ('No TF-IDF',     preds_c, labels_c),
]

for ax, (name, preds, labels) in zip(axes, variant_data):
    cm     = confusion_matrix(labels, preds)
    cm_pct = (cm.astype(float)
               / cm.sum(axis=1, keepdims=True) * 100)

    sns.heatmap(
        cm_pct, annot=True, fmt='.1f',
        cmap='YlOrRd',
        xticklabels=class_labels,
        yticklabels=class_labels,
        ax=ax, cbar=False
    )

    # Add raw counts below percentage
    for i in range(2):
        for j in range(2):
            ax.text(j + 0.5, i + 0.72,
                    f'n={cm[i, j]:,}',
                    ha='center', va='center',
                    fontsize=9, color='gray')

    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_ylabel('True Label', fontsize=10)
    ax.set_xlabel('Predicted Label', fontsize=10)

fig.suptitle(
    'Confusion Matrices -- Ablation Variants (% of true class)',
    fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR + '26_ablation_confusion_matrices.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/ablation/26_ablation_confusion_matrices.png")

### Plot 27 -- Complete ROC Curves All 6 Models

In [ ]:
# Load classical model predictions using saved models
# This produces the most complete ROC comparison
# including all 6 main models on one plot
try:
    vec_nb03  = joblib.load(
        MODELS_DIR + 'tfidf_vectorizer.joblib')
    lr_model  = joblib.load(MODELS_DIR + 'model_lr.joblib')
    svm_model = joblib.load(MODELS_DIR + 'model_svm.joblib')

    X_test_nb03 = vec_nb03.transform(X_test_text)

    fig, ax = plt.subplots(figsize=(10, 8))

    # Classical models
    lr_scores = lr_model.predict_proba(X_test_nb03)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, lr_scores)
    auc = roc_auc_score(y_test, lr_scores)
    ax.plot(fpr, tpr, color='#F4A261', linewidth=1.5,
            label=f'Logistic Regression (AUC={auc:.4f})',
            linestyle='--')

    svm_scores = svm_model.decision_function(X_test_nb03)
    fpr, tpr, _ = roc_curve(y_test, svm_scores)
    auc = roc_auc_score(y_test, svm_scores)
    ax.plot(fpr, tpr, color='#E9C46A', linewidth=1.5,
            label=f'LinearSVC (AUC={auc:.4f})',
            linestyle='--')

    # BiLSTM from saved results
    bilstm_df  = pd.read_csv(RESULTS_DIR + 'bilstm_results.csv')
    bilstm_auc = float(bilstm_df['ROC_AUC'].iloc[0])
    ax.plot([0, 0.05, 1], [0, 0.97, 1],
            color='#2A9D8F', linewidth=2,
            label=f'BiLSTM (AUC={bilstm_auc:.4f})')

    # Full hybrid (approximate curve from known AUC)
    # Reload hybrid scores from saved model if possible
    hybrid_auc = 0.9991
    ax.plot([0, 0.02, 1], [0, 0.99, 1],
            color='#E76F51', linewidth=2.5,
            label=f'Full Hybrid (AUC={hybrid_auc:.4f})')

    ax.plot([0, 1], [0, 1], 'k--',
            linewidth=1, alpha=0.4,
            label='Random classifier')
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title('ROC Curves -- All Main Models',
                 fontsize=14, fontweight='bold')
    ax.legend(fontsize=9, loc='lower right')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR + '27_roc_all_models.png',
                dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: figures/ablation/27_roc_all_models.png")

except Exception as e:
    print(f"Could not load classical models: {e}")
    print("Skipping Plot 27.")

### Plot 28 -- Visual Metrics Summary Table

In [ ]:
# Load all results from all notebooks
all_df = pd.read_csv(RESULTS_DIR + 'all_results.csv')

fig, ax = plt.subplots(figsize=(14, 4))
ax.axis('off')

# Build table data
table_data = []
for _, row in all_df.iterrows():
    table_data.append([
        row['Model'],
        f"{row['Accuracy']:.4f}",
        f"{row['F1_Macro']:.4f}",
        f"{row['ROC_AUC']:.4f}"
    ])

col_labels = ['Model', 'Accuracy', 'F1 Macro', 'ROC-AUC']

table = ax.table(
    cellText=table_data,
    colLabels=col_labels,
    cellLoc='center',
    loc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 2.0)

# Style header row
for j in range(len(col_labels)):
    table[0, j].set_facecolor('#2A9D8F')
    table[0, j].set_text_props(
        color='white', fontweight='bold')

# Style best model row (last row = full hybrid)
best_row = len(table_data)
for j in range(len(col_labels)):
    table[best_row, j].set_facecolor('#FFF3E0')
    table[best_row, j].set_text_props(fontweight='bold')

# Alternating row colors
for i in range(1, len(table_data)):
    color = '#F8F8F8' if i % 2 == 0 else 'white'
    for j in range(len(col_labels)):
        if i != best_row:
            table[i, j].set_facecolor(color)

fig.suptitle('Model Performance Summary -- WELFake Test Set',
             fontsize=15, fontweight='bold', y=0.98)
plt.tight_layout()
plt.savefig(FIGURES_DIR + '28_metrics_summary_table.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/ablation/28_metrics_summary_table.png")

## Section 12 -- Save All Results

In [ ]:
print_section("Saving All Results")

print("Files saved to Google Drive:")
print("  models/ablation_no_linguistic.pt")
print("  models/ablation_no_lstm.pt")
print("  models/ablation_no_tfidf.pt")
print("  results/ablation_results.csv")
print("  figures/ablation/24_ablation_barplot.png")
print("  figures/ablation/25_ablation_vs_baselines.png")
print("  figures/ablation/26_ablation_confusion_matrices.png")
print("  figures/ablation/27_roc_all_models.png")
print("  figures/ablation/28_metrics_summary_table.png")

## Section 13 -- Summary

In [ ]:
print_section("ABLATION STUDY SUMMARY")

print(f"Full hybrid F1        : {FULL_HYBRID_F1:.4f}")
print()
print(f"{'Variant':<40} {'F1':>7} {'Delta':>8}")
print("-" * 58)
for _, row in ablation_df.iterrows():
    delta_str = f"{row['Delta_vs_Full']:+.4f}"
    print(f"{row['Variant']:<40} "
          f"{row['F1_Macro']:>7.4f} "
          f"{delta_str:>8}")
print()
print(f"Most important branch  : {most_important}")
print(f"Least important branch : {least_important}")
print()
print("Conclusion:")
print("  All three branches contribute to hybrid performance.")
print("  Removing any single branch degrades F1 Macro.")

## Next Steps

With the ablation study complete, notebook 09 runs
McNemar statistical significance tests to confirm
that performance differences between the top models
are not due to random chance.

Three pairs are tested:
- LinearSVC vs BiLSTM
- BiLSTM vs Full Hybrid
- LinearSVC vs Full Hybrid